# Qwen2.5 PA3 ? full TRAIN, one fixed shard per saved version
Prepare four notebook versions with SHARD_ID 0, 1, 2, 3. Sizes: 941, 941, 941, 940.
LOCKED until DEC-PA3-007 and the separate TRAIN authority have been reviewed, audited and merged.
Attach pinned Model Prep, Runtime Prep + inventory, approved TRAIN dataset, source bundle, and technical qualification + its original bundle.
T4, Internet OFF. Record the exact shard in the external attempt ledger. Click **Save Version ? Save & Run All** exactly once.
Download PA3_TRAIN_SHARD_<id>_OUTPUT.tar.gz and its SHA-256 from the completed Version ? Output. Failure consumes the attempt; do not rerun.
See notes/pa3_full_train_kaggle_runbook.md. No TEST, threshold, fallback or GPU run by Codex.
A saved Version can contain a FAILED shard: inspect metrics.completed and audit/failure records; saving the notebook does not mean inference passed.

In [ ]:
# Edit SHARD_ID and fill attachment paths/hashes from the audited release, before saving.
SHARD_ID = 0
SETTINGS = {
    "mode": "full-train",
    "shard_id": 0,
    "run_id": "pa3-qwen-full-train-shard-0",
    "source_commit": "",
    "source_bundle_input": "",
    "source_bundle_sha256": "",
    "model_input": "",
    "runtime_input": "",
    "runtime_inventory_input": "",
    "runtime_inventory_sha256": "",
    "runtime_site_packages": "",
    "dataset_input": "",
    "full_manifest_sha256": "",
    "inference_manifest_sha256": "",
    "technical_run_input": "",
    "technical_seal_sha256": "",
    "technical_source_bundle_input": "",
    "technical_source_bundle_sha256": ""
}
SETTINGS["shard_id"] = SHARD_ID
SETTINGS["run_id"] = f"pa3-qwen-full-train-shard-{SHARD_ID}"


In [ ]:
# A: verify trusted committed source before importing attached Python.
# B-G: reuse pinned prep/runtime/T1, audit prior technical run, verify exact TRAIN shard,
# capture raw logits, seal, join GT and export CPU-recomputed metrics.
import hashlib
import json
import os
from pathlib import Path, PurePosixPath
import re
import subprocess
import sys
import tarfile

work = Path.cwd()
inputs = work.parent / "input"
run_id = SETTINGS.get("run_id", "")
if not re.fullmatch(r"pa3-qwen-[a-z0-9][a-z0-9_-]{0,63}", run_id):
    raise ValueError("SAFE_UNIQUE_RUN_ID_REQUIRED")
evidence = work / "data/processed/pa3_bootstrap" / run_id
evidence.mkdir(parents=True, exist_ok=False)
stage = "A_SOURCE_BUNDLE"

def write(name, value):
    payload = (json.dumps(value, sort_keys=True, allow_nan=False) + "\n").encode()
    with (evidence / name).open("xb") as stream:
        stream.write(payload)
        stream.flush()
        os.fsync(stream.fileno())

def digest(path):
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

try:
    sha = SETTINGS["source_commit"]
    if not re.fullmatch(r"[0-9a-f]{40}", sha):
        raise ValueError("REVIEWED_SOURCE_COMMIT_REQUIRED")
    rel = SETTINGS["source_bundle_input"]
    p = PurePosixPath(rel)
    if (not rel or p.is_absolute() or ".." in p.parts or "\\" in rel or ":" in rel):
        raise ValueError("INPUT_RELATIVE_BUNDLE_REQUIRED")
    bundle = inputs / p
    if not bundle.resolve().is_relative_to(inputs.resolve()):
        raise ValueError("BUNDLE_ESCAPES_INPUT")
    if digest(bundle) != SETTINGS["source_bundle_sha256"]:
        raise ValueError("TRUSTED_SOURCE_BUNDLE_HASH_MISMATCH")
    repo = work / "data/processed/pa3_source" / run_id
    if repo.exists():
        raise FileExistsError("SOURCE_EXISTS_NO_RETRY")
    subprocess.run(["git", "-c", "core.autocrlf=false", "clone", "--no-checkout", str(bundle), str(repo)], check=True)
    subprocess.run(["git", "-C", str(repo), "-c", "core.autocrlf=false", "checkout", "--detach", sha], check=True)
    head = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
    if head != sha:
        raise ValueError("SOURCE_COMMIT_MISMATCH")
    write("source.json", {"source_commit": sha, "bundle_sha256": SETTINGS["source_bundle_sha256"]})
    if any(k == "safeshift" or k.startswith("safeshift.") for k in sys.modules):
        raise ValueError("FRESH_NOTEBOOK_KERNEL_REQUIRED")
    sys.path.insert(0, str(repo))
    os.environ.update({"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
                       "HF_HUB_DISABLE_TELEMETRY": "1", "HF_HUB_DISABLE_XET": "1",
                       "HF_HUB_ENABLE_HF_TRANSFER": "0", "CUDA_VISIBLE_DEVICES": "0"})
    from safeshift.pa3.native.kaggle import launch
    from safeshift.runners.internvl3_snapshot import network_denied
    stage = "B_TO_G_AUTHORIZED_WORKFLOW"
    with network_denied():
        output = launch(repo, inputs, SETTINGS)
    write("finish.json", {"status": "WORKFLOW_COMPLETE", "artifacts": output.relative_to(work).as_posix()})
    print("PA3 artifacts:", output.relative_to(work))
except BaseException as exc:
    write("failure.json", {"stage": stage, "exception_type": type(exc).__name__,
                           "retry_permitted": False, "runtime_pass_claim": False})
    # Finish the saved Version so failure artifacts are retained in Output.
    # This changes no run status or retry policy; the exception is recorded above.
    print("SHARD FAILED; preserve this attempt and do not rerun:", type(exc).__name__, str(exc))
finally:
    write("checksums.json", {p.name: {"sha256": digest(p), "size_bytes": p.stat().st_size}
                              for p in sorted(evidence.iterdir()) if p.is_file()})
    archive = evidence.with_suffix(".tar.gz")
    with tarfile.open(archive, "x:gz") as stream:
        stream.add(evidence, arcname=evidence.name)
    with archive.with_suffix(".sha256.json").open("x", encoding="utf-8") as stream:
        json.dump({"archive": archive.name, "sha256": digest(archive)}, stream)
    print("Bootstrap evidence:", archive.relative_to(work))
    # Runs live below /kaggle/working throughout inference; a kernel failure cannot
    # be repaired by rerunning. Preserve any partial files the Version retains.
    stage = "SAVE_VERSION_OUTPUT_PACKAGE"
    artifact_root = repo / "data/processed/pa3_qwen" if "repo" in globals() else None
    output_archive = work / f"PA3_TRAIN_SHARD_{SHARD_ID}_OUTPUT.tar.gz"
    with tarfile.open(output_archive, "x:gz") as stream:
        stream.add(evidence, arcname="bootstrap")
        if artifact_root and artifact_root.exists():
            for entry in sorted(artifact_root.iterdir()):
                if entry.name.startswith(run_id) and "technical-source" not in entry.name:
                    stream.add(entry, arcname=entry.name)
    with output_archive.with_suffix(".sha256.json").open("x", encoding="utf-8") as stream:
        json.dump({"archive": output_archive.name, "sha256": digest(output_archive),
                   "run_id": run_id, "shard_id": SHARD_ID, "retry_permitted": False}, stream)
    print("Download from this saved Version / Output:", output_archive.name)
